# Chạy tự động toàn bộ dự án Dubbing trên Google Colab (1 Click)

Nhấn **Run** ở ô Code duy nhất bên dưới để tự động thực hiện toàn bộ quy trình:
1. Clone repository từ GitHub.
2. Cài đặt FFmpeg, Node.js và Cloudflared CLI.
3. Cài đặt Node.js dependencies (`npm install`) và Python dependencies (`pip install`).
4. Tự động chạy `device_register.py` lấy `DUANJU_DEVICE_ID` và `DUANJU_INSTALL_ID` động để tạo file `.env` & `settings.json`.
5. Khởi chạy FastAPI Backend Server và mở Public URL qua **Cloudflare Tunnel (trycloudflare.com)**.

In [ ]:
# -----------------------------------------------------
# Quy trình thiết lập & chạy tự động 1-Click trên Colab
# -----------------------------------------------------
import os
import sys
import json
import time
import re
import subprocess

print("🚀 [1/5] Clone Repository từ GitHub...")
!rm -rf /content/dubbing
!git clone https://github.com/kinyias/dubbing.git /content/dubbing
os.chdir("/content/dubbing")

print("\n📦 [2/5] Cài đặt System Dependencies (FFmpeg, Node.js, Cloudflared)...")
!apt-get update -qq
!apt-get install -y ffmpeg nodejs npm wget -qq
!if ! which cloudflared >/dev/null 2>&1; then \
    wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && \
    chmod +x /usr/local/bin/cloudflared; \

print("\n🐍 [3/5] Cài đặt Node.js & Python dependencies...")
!cd /content/dubbing/backend && npm install
!pip install -r /content/dubbing/requirements.txt

print("\n📲 [4/5] Tự động đăng ký thiết bị lấy Device ID & Install ID động...")
backend_path = os.path.abspath("/content/dubbing/backend")
if backend_path not in sys.path:
    sys.path.insert(0, backend_path)
sys.path.insert(0, os.path.join(backend_path, "service", "liushen"))

device_id = ""
install_id = ""
try:
    from service.liushen.device_register import device_register
    reg_res = device_register()
    device_id = reg_res.get("device_id", "")
    install_id = reg_res.get("install_id", "")
    print(f"✅ Đăng ký thiết bị thành công! Device ID: {device_id} | Install ID: {install_id}")
except Exception as e:
    print(f"⚠️ Lỗi đăng ký thiết bị: {e}")

# Tạo file .env
env_content = f"""DUANJU_DEVICE_ID={device_id}
DUANJU_INSTALL_ID={install_id}
DUANJU_PLATFORM=android
APP_PORT=8000
OPEN_BROWSER=0
FLASK_DEBUG=0
FFMPEG_BIN=ffmpeg
"""
with open("/content/dubbing/.env", "w", encoding="utf-8") as f:
    f.write(env_content)

# Tạo file backend/settings.json
settings_example_path = "/content/dubbing/backend/settings.example.json"
settings_path = "/content/dubbing/backend/settings.json"
if os.path.exists(settings_example_path):
    with open(settings_example_path, "r", encoding="utf-8") as f:
        settings = json.load(f)
    settings["ffmpegPath"] = "ffmpeg"
    with open(settings_path, "w", encoding="utf-8") as f:
        json.dump(settings, f, indent=4, ensure_ascii=False)
print("✅ Khởi tạo cấu hình .env và settings.json hoàn tất!")

print("\n🌐 [5/5] Khởi chạy Backend Server & Mở Cloudflare Tunnel...")
backend_proc = subprocess.Popen(["python", "/content/dubbing/backend/main.py"])
time.sleep(3)

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("🚀 Đang chờ Cloudflare cấp Public API URL...")
for line in iter(tunnel_proc.stdout.readline, ""):
    print(line, end="")
    if "trycloudflare.com" in line:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            print("\n==================================================")
            print(f"🎉 PUBLIC API URL: {match.group(0)}")
            print("==================================================\n")